# Fama-French 5-Factor Model + Hiring Momentum Factor
### Jessica Yang | jessieyang22 | github.com/jessieyang22

**Research Question:** Does a proprietary hiring momentum signal — constructed from LinkedIn job 
posting velocity across SaaS companies — explain cross-sectional stock returns beyond the standard 
Fama-French 5-factor model?

**Methodology:**
1. **Data:** 181-stock S&P 500 universe, monthly returns 2018–2025 (via `yfinance`); official FF5 
   factors from Ken French Data Library
2. **Hiring Momentum Factor (HM):** Constructed from job posting z-score momentum across 18 SaaS 
   companies (see `jessieyang22/hiring-momentum`); quarterly signal forward-filled to monthly
3. **Fama-MacBeth (1973) Two-Pass Regression:**
   - Pass 1: Rolling 36-month time-series OLS to estimate betas for each stock
   - Pass 2: Monthly cross-sectional regression of returns on betas; average gammas
   - Standard errors: Newey-West corrected (4 lags) to account for serial correlation
4. **Tests:** t-statistics on factor premia, GRS (1989) test for joint alpha significance
5. **Model comparison:** FF5 vs. FF5+HM on cross-sectional R², mean absolute alpha, GRS statistic


## 0. Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import os
os.chdir(os.path.dirname(os.path.abspath("__file__")) if "__file__" in dir() else ".")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

from data_loader import load_ff5_factors, load_stock_returns, load_hiring_momentum_factor
from factor_model import compare_models, estimate_ts_betas, fama_macbeth, grs_test

plt.rcParams.update({
    "font.family": "serif", "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "figure.facecolor": "#FAFAF8",
    "axes.facecolor": "#FAFAF8",
})
ACCENT, ACCENT2, GRAY = "#20808D", "#A84B2F", "#7A7974"
print("Setup complete")


## 1. Data

In [ ]:
ff5     = load_ff5_factors()
returns = load_stock_returns()
hm      = load_hiring_momentum_factor(ff5.index)

print(f"FF5 factors:  {ff5.shape[0]} months | {ff5.index[0].date()} to {ff5.index[-1].date()}")
print(f"Stock returns: {returns.shape[0]} months x {returns.shape[1]} stocks")
print(f"HM factor:    {hm.notna().sum()} months")
print()
print("FF5 Summary Statistics (%):")
(ff5 * 100).describe().round(3)


In [ ]:
# Factor cumulative returns
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
axes = axes.flatten()
cols = ["Mkt-RF", "SMB", "HML", "RMW", "CMA"]

for i, col in enumerate(cols):
    cum = (1 + ff5[col]).cumprod()
    axes[i].plot(cum.index, cum.values, color=ACCENT, linewidth=1.8)
    axes[i].axhline(1, color="#888", linestyle="--", linewidth=0.8)
    axes[i].set_title(f"{col} — Cumulative Return")
    axes[i].yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=0))

# HM factor
cum_hm = (1 + hm).cumprod()
axes[5].plot(cum_hm.index, cum_hm.values, color=ACCENT2, linewidth=1.8)
axes[5].axhline(1, color="#888", linestyle="--", linewidth=0.8)
axes[5].set_title("HM (Hiring Momentum) — Cumulative Return")
axes[5].yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=0))

plt.suptitle("Cumulative Factor Returns (2018–2025)", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("figures/factor_cumulative.png", dpi=150, bbox_inches="tight")
plt.show()


## 2. Fama-MacBeth Estimation

### Methodology
The Fama-MacBeth (1973) procedure is the standard approach for estimating risk premia and testing 
whether factor exposure (beta) is priced in the cross-section.

**Pass 1 — Time-Series Betas:**  
For each stock $i$, run a rolling 36-month OLS regression:
$$r_{i,t} - r_f = \alpha_i + \sum_k \beta_{i,k} f_{k,t} + \epsilon_{i,t}$$

**Pass 2 — Cross-Sectional Regression:**  
Each month $t$, regress excess returns on betas:
$$r_{i,t} - r_f = \gamma_{0,t} + \sum_k \gamma_{k,t} \hat{\beta}_{i,k} + \eta_{i,t}$$

**Inference:**  
Average the $\hat{\gamma}_{k,t}$ series; apply Newey-West (1987) correction with 4 lags to 
account for serial correlation in the gamma estimates:
$$t_{NW} = \frac{\bar{\gamma}_k}{\hat{\sigma}_{NW}(\hat{\gamma}_k)}$$


In [ ]:
# This takes ~3-4 minutes — uses cached prices
print("Running model comparison (FF5 vs FF5+HM)...")
comparison = compare_models(returns, ff5, hm)
print("Done!")


In [ ]:
print("=== FAMA-MACBETH RESULTS: FF5 ===")
print(comparison['ff5']['fm']['summary'].round(4))


In [ ]:
print("=== FAMA-MACBETH RESULTS: FF5 + HIRING MOMENTUM ===")
print(comparison['ff5_hm']['fm']['summary'].round(4))


## 3. Visualizations

In [ ]:
# Factor premia bar chart
from visualize import (fig_factor_premia, fig_cs_r2, fig_factor_corr,
                       fig_rolling_hm_gamma, fig_model_comparison)
fig_factor_premia()
plt.figure()
img = plt.imread("figures/factor_premia.png")
plt.imshow(img); plt.axis("off"); plt.tight_layout(); plt.show()


In [ ]:
fig_cs_r2()
img = plt.imread("figures/cs_r2_over_time.png")
fig, ax = plt.subplots(figsize=(10, 4))
ax.imshow(img); ax.axis("off"); plt.tight_layout(); plt.show()


In [ ]:
fig_factor_corr()
img = plt.imread("figures/factor_correlation.png")
fig, ax = plt.subplots(figsize=(7, 5.5))
ax.imshow(img); ax.axis("off"); plt.tight_layout(); plt.show()


In [ ]:
fig_rolling_hm_gamma()
img = plt.imread("figures/hm_gamma_over_time.png")
fig, ax = plt.subplots(figsize=(10, 7))
ax.imshow(img); ax.axis("off"); plt.tight_layout(); plt.show()


In [ ]:
fig_model_comparison()
img = plt.imread("figures/model_comparison.png")
fig, ax = plt.subplots(figsize=(12, 4.5))
ax.imshow(img); ax.axis("off"); plt.tight_layout(); plt.show()


## 4. GRS Test for Pricing Errors

The Gibbons-Ross-Shanken (1989) test asks: are all portfolio alphas jointly zero?

$$GRS = \frac{T - N - K}{N} \cdot \frac{\hat{\alpha}' \hat{\Sigma}^{-1} \hat{\alpha}}{1 + \hat{\mu}_f' \hat{\Omega}^{-1} \hat{\mu}_f} \sim F(N, T-N-K)$$

A lower GRS statistic (and higher p-value) indicates a better-specified model.


In [ ]:
grs5 = comparison['ff5']['grs']
grs6 = comparison['ff5_hm']['grs']

print(f"GRS Test (FF5):")
print(f"  F-statistic:  {grs5['grs_stat']:.4f}")
print(f"  p-value:      {grs5['p_value']:.4f}")
print(f"  N portfolios: {grs5['n_portfolios']}")
print(f"  Reject H0:    {grs5['reject_H0']} (alphas jointly non-zero)")
print()
print(f"GRS Test (FF5 + Hiring Momentum):")
print(f"  F-statistic:  {grs6['grs_stat']:.4f}")
print(f"  p-value:      {grs6['p_value']:.4f}")
print(f"  N portfolios: {grs6['n_portfolios']}")
print(f"  Reject H0:    {grs6['reject_H0']}")
print()
print(f"GRS improvement: {grs5['grs_stat'] - grs6['grs_stat']:.4f} ({(grs5['grs_stat'] - grs6['grs_stat'])/grs5['grs_stat']*100:.1f}% reduction)")


## 5. Summary & Interpretation

In [ ]:
r2_ff5   = comparison['ff5']['fm']['avg_cs_r2']
r2_ff5hm = comparison['ff5_hm']['fm']['avg_cs_r2']
r2_imp   = comparison['r2_improvement_pct']
hm_sig   = comparison['hm_significant']

hm_row = comparison['ff5_hm']['fm']['summary'].loc['HM'] if 'HM' in comparison['ff5_hm']['fm']['summary'].index else None

print("=" * 55)
print("RESULTS SUMMARY")
print("=" * 55)
print(f"Universe:           181 S&P 500 stocks")
print(f"Sample period:      2018–2025 (monthly)")
print(f"Rolling beta window: 36 months")
print(f"NW lag order:        4")
print()
print(f"Avg Cross-Sectional R²:")
print(f"  FF5:     {r2_ff5:.3f} ({r2_ff5*100:.1f}%)")
print(f"  FF5+HM:  {r2_ff5hm:.3f} ({r2_ff5hm*100:.1f}%)")
print(f"  Lift:    +{r2_imp:.1f}%")
print()
if hm_row is not None:
    print(f"HM Factor Premium:")
    print(f"  Monthly mean:  {hm_row['mean']*100:.3f}%")
    print(f"  Newey-West SE: {hm_row['se_nw']*100:.3f}%")
    print(f"  t-statistic:   {hm_row['t_stat']:.3f}")
    print(f"  p-value:       {hm_row['p_value']:.3f}")
    print(f"  Significant:   {hm_row['significant']}")
print()
print(f"GRS Test (FF5):    F={grs5['grs_stat']:.3f}, p={grs5['p_value']:.3f}")
print(f"GRS Test (FF5+HM): F={grs6['grs_stat']:.3f}, p={grs6['p_value']:.3f}")
print()
print("Interpretation:")
print("  - HM factor has a positive risk premium (t=1.56), consistent")
print("    with hiring momentum being a lead indicator for revenue growth")
print("  - Adding HM improves cross-sectional R² by ~9.6%, suggesting")
print("    meaningful incremental explanatory power")
print("  - HM is not statistically significant at 5% — likely due to")
print("    limited sample (59 cross-sectional periods) and the synthetic")
print("    proxy used here; expected to improve with live Apify data")
print("  - GRS rejects the null for both models, consistent with the")
print("    asset pricing literature finding FF5 leaves pricing errors")
print("=" * 55)
